# 00 · Core functions (shared by all notebooks)

This notebook only **defines** functions and settings; it runs no experiment.
Every section notebook loads it with

```python
%run ./00_core.ipynb
```

Contents

1. **Settings** – copy numbers, shots, seeds, qubit layouts on `FakeKingston`.
2. **Theory** – the vectors $\lvert r_i\rangle$ (Eq. 2) and the unitary completion $\mathcal W$ (Eq. 3).
3. **Measurement circuits** – QSD and isometry decompositions of $\mathcal W$.
4. **Protocol circuits** – teleportation / program retrieval for $m=1$ and $m=2$.
5. **Execution** – ideal `AerSimulator` and noisy `FakeKingston` simulator.
6. **Analysis** – post-selection on Alice's herald, success probability, fidelity.

Qubit order (Qiskit little-endian) for $m=1$: `copy_0 … copy_{k-1}, A, B`.

In [1]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------
from itertools import combinations
from math import comb, ceil, log2

import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.circuit.library import UnitaryGate, Isometry, XGate, YGate
from qiskit.quantum_info import Operator, Statevector
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import ALAPScheduleAnalysis, PadDelay, PadDynamicalDecoupling

from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime.fake_provider import FakeKingston   # offline snapshot, no IBM account needed

In [2]:
# ---------------------------------------------------------------------------
# Global settings (change here, every notebook follows)
# ---------------------------------------------------------------------------
D = 2                          # local dimension (qubits)
KS = (1, 2, 3, 4, 5)           # copy numbers k used in the paper
METHODS = ("qsd", "isometry")  # the two decompositions of the measurement unitary
SHOTS = 10_000                 # shots per circuit (as in the paper)
SEED = 1234                    # fixed seeds -> reproducible transpilation and sampling

# Physical qubits on ibm_kingston, ordered as the circuit qubits [copies..., A, B].
# Layouts are nested: going from k to k+1 only adds one copy qubit, so the
# k-dependence is not polluted by switching to different hardware qubits.
LAYOUT_1 = {1: [49, 38, 29],
            2: [49, 48, 38, 29],
            3: [49, 48, 50, 38, 29],
            4: [49, 48, 50, 47, 38, 29],
            5: [49, 48, 50, 47, 51, 38, 29]}
# Second block for m = 2 (two-qubit programs). B_1 = 29 and B_2 = 28 are coupled,
# so the two-qubit program between Bob's qubits needs no routing.
LAYOUT_2 = {1: [26, 27, 28],
            2: [26, 17, 27, 28],
            3: [26, 17, 25, 27, 28],
            4: [26, 17, 25, 7, 27, 28],
            5: [26, 17, 25, 7, 24, 27, 28]}

# Backends
FAKE = FakeKingston()                          # 156-qubit Heron r2 snapshot (gates, readout, T1/T2)
NOISE_MODEL = NoiseModel.from_backend(FAKE)    # also adds T1/T2 relaxation to delay instructions
SIM_IDEAL = AerSimulator()
SIM_NOISY = AerSimulator(noise_model=NOISE_MODEL)
RNG = np.random.default_rng(SEED)              # draws a new simulator seed for every run


def p_theory(k, d=D):
    # Optimal success probability p(d, k) = k / [d (k - 1 + d)]
    return k / (d * (k - 1 + d))


print(f"FakeKingston: {FAKE.num_qubits} qubits, native gates {sorted(FAKE.operation_names)}")

FakeKingston: 156 qubits, native gates ['cz', 'delay', 'id', 'if_else', 'measure', 'measure_2', 'reset', 'rz', 'sx', 'x']


## Theory: measurement vectors and unitary completion

The optimal POVM element is $\hat M=\sum_{i=0}^{R-1}\lvert r_i\rangle\langle r_i\rvert$ with
$R=\binom{k-2+d}{k-1}$ (Eq. 1) and

$$\lvert r_i\rangle=\frac{1}{\gamma_{k,d}}\sum_{a=1}^{k}\lvert D^{(k-1)}_i\rangle_{[k]\setminus a}\otimes\lvert\Phi^+_d\rangle_{aA},\qquad \gamma_{k,d}=\sqrt{k(k-1+d)/d}\quad\text{(Eq. 2)}.$$

To measure $\{\hat M,\mathbb 1-\hat M\}$ we complete $\mathcal M=[\lvert r_0\rangle\cdots\lvert r_{R-1}\rangle]$
to a unitary $\mathcal W$ (Eq. 3) with a full QR decomposition. After $\mathcal W$, a computational-basis
outcome $<R$ on Alice's $k+1$ qubits means **success**.

In [3]:
def dicke_state(n, w):
    # |D^n_w>: equal superposition of all n-bit strings of Hamming weight w.
    # The Dicke states {w = 0..n} form an orthonormal basis of the symmetric subspace.
    if n == 0:
        return np.array([1.0])
    v = np.zeros(2 ** n)
    for ones in combinations(range(n), w):
        v[sum(1 << q for q in ones)] = 1.0
    return v / np.linalg.norm(v)


def apply_swap(vec, q1, q2):
    # SWAP of qubits q1 and q2 acting on a statevector (permutes the index bits).
    out = np.zeros_like(vec)
    for idx in range(len(vec)):
        b1, b2 = (idx >> q1) & 1, (idx >> q2) & 1
        out[idx if b1 == b2 else idx ^ ((1 << q1) | (1 << q2))] = vec[idx]
    return out


def build_r_matrix(k, d=D):
    # Columns are the vectors |r_i> of Eq. (2) on k+1 qubits (copies 0..k-1, A = qubit k).
    # Construction: put the Dicke state on copies 0..k-2 and |Phi+> on (copy k-1, A),
    # then swap copy k-1 with every copy a -> this realises the sum over a in Eq. (2).
    R = comb(k - 2 + d, k - 1)
    bell = np.array([1, 0, 0, 1]) / np.sqrt(2)
    gamma = np.sqrt(k * (k - 1 + d) / d)
    cols = []
    for i in range(R):
        base = np.kron(bell, dicke_state(k - 1, i))
        r = sum(apply_swap(base, a, k - 1) for a in range(k))
        cols.append(r / gamma)
    M = np.array(cols).T
    assert np.allclose(M.T @ M, np.eye(R), atol=1e-12), "|r_i> are not orthonormal"
    return M, R


def unitary_completion(M):
    # Eq. (3): full QR decomposition M = Q R_qr. Fix column signs so that the first R
    # columns of Q equal M; the remaining columns span the complement.  W = Q^dagger.
    N, R = M.shape
    Q, Rqr = np.linalg.qr(M, mode="complete")
    Q[:, :R] = Q[:, :R] * np.sign(np.diag(Rqr[:R, :R]))
    assert np.allclose(Q[:, :R], M, atol=1e-10)
    W = Q.conj().T
    assert np.allclose(W @ W.conj().T, np.eye(N), atol=1e-10)
    return W

## Measurement circuits: QSD vs isometry

* **`qsd`** – the full $2^{k+1}\times2^{k+1}$ unitary $\mathcal W$ is handed to the transpiler,
  which synthesises it with the quantum Shannon decomposition ($\Theta(4^{k})$ two-qubit gates).
* **`isometry`** – only the physical columns are synthesised: the first $2^{m}$ columns of
  $\mathcal W^\dagger$ ($m=\lceil\log_2 R\rceil$, padding taken from the complement), then inverted
  ($O(k\,2^{k})$ two-qubit gates). Same POVM, shallower circuit.

In [4]:
_MEAS_CACHE = {}   # synthesis is the slow step -> build each (k, method) only once


def measurement_circuit(k, method):
    # Returns (circuit on k+1 qubits implementing W, R). Outcome < R  <=>  success.
    if (k, method) in _MEAS_CACHE:
        return _MEAS_CACHE[(k, method)]
    M, R = build_r_matrix(k)
    W = unitary_completion(M)
    n = k + 1

    if method == "qsd":
        sub = QuantumCircuit(n, name=f"W_qsd(k={k})")
        sub.append(UnitaryGate(W, label=f"W(k={k})"), range(n))   # transpiler applies QSD

    elif method == "isometry":
        m = max(1, ceil(log2(R)))
        cols = W.conj().T[:, : 2 ** m]          # |r_0..r_{R-1}> + padding from the complement
        sub = None
        # Qiskit's Isometry qubit-order convention changed between versions: try both
        # orders and keep the one whose columns really are the |r_i> (up to phases).
        for order in (list(range(n)), list(reversed(range(n)))):
            iso = QuantumCircuit(n)
            iso.append(Isometry(cols, num_ancillas_zero=0, num_ancillas_dirty=0), order)
            iso = iso.decompose(reps=6)
            G = Operator(iso).data
            if all(abs(abs(np.vdot(M[:, i], G[:, i])) - 1) < 1e-8 for i in range(R)):
                sub = iso.inverse()             # isometry prepares |r_i>; inverse measures it
                break
        if sub is None:
            raise RuntimeError("isometry synthesis does not reproduce |r_i>")
        sub.name = f"W_iso(k={k})"
    else:
        raise ValueError(f"unknown method {method!r}")

    _MEAS_CACHE[(k, method)] = (sub, R)
    return sub, R

## Protocol circuits

Storage → (optional memory delay) → retrieval, exactly as in Fig. 1:

1. Bell pair $\lvert\Phi^+\rangle_{AB}$; the stored program $\hat U$ acts once on Bob's qubit.
2. Optional delay $\tau$ on $A$ and $B$ (memory lifetime, Fig. 4).
3. $k$ copies of the input, the measurement $\mathcal W$ on copies $+A$, readout of Alice (herald).
4. Bob: single-qubit tomography ($m=1$) or undo $\hat U$ and read out ($m=2$).

Plain teleportation is the special case $\hat U=\mathbb 1$.

In [5]:
# Single-qubit input states used in the paper and their kets
STATE_KETS = {
    "0": np.array([1, 0]),
    "1": np.array([0, 1]),
    "+": np.array([1, 1]) / np.sqrt(2),
    "-": np.array([1, -1]) / np.sqrt(2),
    "+i": np.array([1, 1j]) / np.sqrt(2),
    "-i": np.array([1, -1j]) / np.sqrt(2),
}

# Single-qubit programs: gates applied to Bob during storage, and their matrices
PROGRAMS_1Q = {
    "I": ([], np.eye(2)),
    "X": (["x"], np.array([[0, 1], [1, 0]])),
    "H": (["h"], np.array([[1, 1], [1, -1]]) / np.sqrt(2)),
    "Z": (["z"], np.diag([1, -1])),
    "S": (["s"], np.diag([1, 1j])),
    "T": (["t"], np.diag([1, np.exp(1j * np.pi / 4)])),
}


def prepare_state(qc, q, label):
    # Prepare one copy of the input state on qubit q.
    gates = {"0": [], "1": ["x"], "+": ["h"], "-": ["x", "h"],
             "+i": ["h", "s"], "-i": ["h", "sdg"]}[label]
    for g in gates:
        getattr(qc, g)(q)


def single_qubit_circuits(k, method, in_state="+", program="I", delay_us=None):
    # m = 1: three circuits that read Bob in the Z, X and Y basis (state tomography).
    # Returns ({basis: circuit}, R).
    sub, R = measurement_circuit(k, method)
    A, B = k, k + 1
    circuits = {}
    for basis in "ZXY":
        qc = QuantumCircuit(QuantumRegister(k, "copy"), QuantumRegister(1, "A"),
                            QuantumRegister(1, "B"),
                            ClassicalRegister(k + 1, "m_alice"), ClassicalRegister(1, "m_bob"))
        # 1. storage: Bell pair, then the program acts once on Bob's half
        qc.h(A); qc.cx(A, B)
        for g in PROGRAMS_1Q[program][0]:
            getattr(qc, g)(B)
        # 2. memory: the stored program waits on A and B
        if delay_us:
            qc.delay(delay_us, A, unit="us"); qc.delay(delay_us, B, unit="us")
        qc.barrier()
        # 3. retrieval: k copies of the input + collective measurement + herald
        for q in range(k):
            prepare_state(qc, q, in_state)
        qc.compose(sub, qubits=list(range(k + 1)), inplace=True)
        qc.measure(list(range(k + 1)), qc.cregs[0])
        # 4. Bob tomography
        if basis == "X":
            qc.h(B)
        elif basis == "Y":
            qc.sdg(B); qc.h(B)
        qc.measure(B, qc.cregs[1])
        circuits[basis] = qc
    return circuits, R


def program_2q(qc, b1, b2, name, inverse=False):
    # Two-qubit programs of Fig. 2b, stored on Bob's qubits B_1 B_2.
    #   "Bell" : H then CNOT                (|00> -> (|00>+|11>)/sqrt2)
    #   "W"    : H x H then CZ              (as in the experiment; |00> -> (|00>+|01>+|10>-|11>)/2)
    #   "SWAP" : SWAP                       (|01> -> |10>)
    if name == "Bell":
        if not inverse: qc.h(b1); qc.cx(b1, b2)
        else:           qc.cx(b1, b2); qc.h(b1)
    elif name == "W":
        if not inverse: qc.h(b1); qc.h(b2); qc.cz(b1, b2)
        else:           qc.cz(b1, b2); qc.h(b1); qc.h(b2)
    elif name == "SWAP":
        qc.swap(b1, b2)
    else:
        raise ValueError(name)


def two_qubit_circuit(k, method, program, inputs):
    # m = 2 with a product input |inputs[0]> x |inputs[1]>: the measurement factorises,
    # one block per qubit.  Bob undoes the program before readout, so the fidelity is the
    # probability of reading back the input bitstring.  Returns (circuit, R).
    sub, R = measurement_circuit(k, method)
    c1, A1, B1 = QuantumRegister(k, "copy_1"), QuantumRegister(1, "A_1"), QuantumRegister(1, "B_1")
    c2, A2, B2 = QuantumRegister(k, "copy_2"), QuantumRegister(1, "A_2"), QuantumRegister(1, "B_2")
    cregs = [ClassicalRegister(k + 1, "m_alice_1"), ClassicalRegister(1, "m_bob_1"),
             ClassicalRegister(k + 1, "m_alice_2"), ClassicalRegister(1, "m_bob_2")]
    qc = QuantumCircuit(c1, A1, B1, c2, A2, B2, *cregs)
    # storage: two Bell pairs, then the two-qubit program on Bob's qubits
    qc.h(A1[0]); qc.cx(A1[0], B1[0])
    qc.h(A2[0]); qc.cx(A2[0], B2[0])
    program_2q(qc, B1[0], B2[0], program)
    qc.barrier()
    # retrieval: one measurement block per qubit
    for q in c1: prepare_state(qc, q, inputs[0])
    for q in c2: prepare_state(qc, q, inputs[1])
    qc.compose(sub, qubits=list(c1) + [A1[0]], inplace=True)
    qc.compose(sub, qubits=list(c2) + [A2[0]], inplace=True)
    # Bob: undo the program, then read out
    program_2q(qc, B1[0], B2[0], program, inverse=True)
    qc.measure(list(c1) + [A1[0]], cregs[0]); qc.measure(B1, cregs[1])
    qc.measure(list(c2) + [A2[0]], cregs[2]); qc.measure(B2, cregs[3])
    return qc, R

## Execution

* `mode="ideal"` – noiseless `AerSimulator` (checks the theory).
* `mode="noisy"` – circuits transpiled onto `FakeKingston` with the fixed layout and simulated
  with its noise model (gate errors, readout errors, T1/T2 relaxation on delays).
* `schedule="dd"` inserts an XY4 dynamical-decoupling sequence in every idle window;
  `schedule="pad"` fills the same windows with plain delays (fair no-DD reference).
  Heron has no native Y, so Y is translated to native gates after insertion.

In [6]:
def _schedule(circuit, schedule):
    # Insert delays ("pad") or XY4 dynamical decoupling ("dd") into idle windows.
    durations = FAKE.target.durations()
    durations.update([("y", (q,), FAKE.target["x"][(q,)].duration, "s")
                      for q in range(FAKE.num_qubits)])          # Y lasts as long as X
    if schedule == "pad":
        pm = PassManager([ALAPScheduleAnalysis(durations), PadDelay(durations=durations)])
    else:
        pm = PassManager([ALAPScheduleAnalysis(durations),
                          PadDynamicalDecoupling(durations,
                                                 dd_sequence=[XGate(), YGate(), XGate(), YGate()])])
    return transpile(pm.run(circuit), basis_gates=["rz", "sx", "x", "cz"], optimization_level=0)


def transpile_circuits(circuits, mode, layout=None, schedule=None, opt_level=3):
    # Ideal: compile for AerSimulator.  Noisy: compile for FakeKingston on `layout`.
    if mode == "ideal":
        return transpile(circuits, SIM_IDEAL, optimization_level=opt_level, seed_transpiler=SEED)
    tqc = transpile(circuits, backend=FAKE, initial_layout=layout,
                    optimization_level=opt_level, seed_transpiler=SEED)
    if schedule in ("pad", "dd"):
        tqc = [_schedule(c, schedule) for c in tqc]
    return tqc


def run_circuits(circuits, mode, shots=SHOTS, layout=None, schedule=None):
    # Run a list of circuits; returns a list of bitstring lists (one string per shot).
    # Qiskit prints registers last-to-first, separated by spaces.
    tqc = transpile_circuits(circuits, mode, layout, schedule)
    sim = SIM_IDEAL if mode == "ideal" else SIM_NOISY
    result = sim.run(tqc, shots=shots, memory=True,
                     seed_simulator=int(RNG.integers(2**31))).result()
    return [result.get_memory(i) for i in range(len(tqc))]

## Analysis

**Success probability** – fraction of shots with Alice's outcome $<R$ (both blocks for $m=2$).

**Fidelity** ($m=1$) – Bob's Bloch vector $\vec r$ from the post-selected $X,Y,Z$ data, projected into the
Bloch ball, and $\mathcal F=\langle t\rvert\rho\lvert t\rangle=(1+\vec r\cdot\vec r_t)/2$ for the pure target
$\lvert t\rangle=\hat U\lvert\psi\rangle$. **Fidelity** ($m=2$) – probability of reading back the input after
undoing $\hat U$. Error bars are one standard deviation from shot noise.

In [7]:
def bloch_vector(ket):
    # Bloch vector (x, y, z) of a pure single-qubit state.
    a, b = ket
    return np.array([2 * np.real(np.conj(a) * b), 2 * np.imag(np.conj(a) * b),
                     abs(a) ** 2 - abs(b) ** 2])


def analyse_single_qubit(memories, R, shots, target_ket):
    # memories: dict basis -> list of shots 'bob alice'.  Returns p, F and their errors.
    ps, r, r_err = [], [], []
    for basis in "XYZ":
        n_succ = n0 = 0
        for shot in memories[basis]:
            bob, alice = shot.split()
            if int(alice, 2) < R:                  # Alice heralds success
                n_succ += 1
                n0 += bob == "0"
        ps.append(n_succ / shots)
        e = (2 * n0 - n_succ) / n_succ if n_succ else 0.0     # <sigma> = P(0) - P(1)
        r.append(e)
        r_err.append(np.sqrt(max(1 - e ** 2, 0) / n_succ) if n_succ else 0.0)
    r = np.array(r)
    if np.linalg.norm(r) > 1:                      # keep the state physical
        r = r / np.linalg.norm(r)
    rt = bloch_vector(target_ket)
    p = float(np.mean(ps))
    return dict(p=p, p_err=float(np.sqrt(p * (1 - p) / (3 * shots))),
                F=float(0.5 * (1 + r @ rt)),
                F_err=float(0.5 * np.sqrt(np.sum((rt * np.array(r_err)) ** 2))))


def run_single_qubit(k, method, mode, in_state="+", program="I", delay_us=None,
                     shots=SHOTS, schedule=None, layout=None):
    # Build, run and analyse one m = 1 setting.  Target state = U|psi>.
    circuits, R = single_qubit_circuits(k, method, in_state, program, delay_us)
    layout = LAYOUT_1[k] if layout is None else layout
    mems = run_circuits([circuits[b] for b in "ZXY"], mode, shots,
                        layout=layout, schedule=schedule)
    target = PROGRAMS_1Q[program][1] @ STATE_KETS[in_state]
    return analyse_single_qubit(dict(zip("ZXY", mems)), R, shots, target)


def run_two_qubit(k, method, mode, program, inputs, shots=SHOTS):
    # Build, run and analyse one m = 2 setting (both blocks must herald success).
    qc, R = two_qubit_circuit(k, method, program, inputs)
    (mem,) = run_circuits([qc], mode, shots, layout=LAYOUT_1[k] + LAYOUT_2[k])
    n_succ = n_ok = 0
    for shot in mem:
        bob2, alice2, bob1, alice1 = shot.split()      # registers print in reverse order
        if int(alice1, 2) < R and int(alice2, 2) < R:
            n_succ += 1
            n_ok += (bob1, bob2) == (inputs[0], inputs[1])
    p = n_succ / shots
    F = n_ok / n_succ if n_succ else 0.0
    return dict(p=p, p_err=float(np.sqrt(p * (1 - p) / shots)),
                F=F, F_err=float(np.sqrt(F * (1 - F) / n_succ)) if n_succ else 0.0)


def exact_check(k, method, trials=5, seed=SEED):
    # Noiseless statevector check of the protocol for random input states:
    # p must equal p(2,k) for every input, and Bob's heralded state must be exact.
    rng = np.random.default_rng(seed)
    sub, R = measurement_circuit(k, method)
    ps, worst_F = [], 1.0
    for _ in range(trials):
        th, ph = rng.uniform(0, np.pi), rng.uniform(0, 2 * np.pi)
        psi = np.array([np.cos(th / 2), np.exp(1j * ph) * np.sin(th / 2)])
        qc = QuantumCircuit(k + 2)
        qc.h(k); qc.cx(k, k + 1)
        for q in range(k):
            qc.ry(th, q); qc.rz(ph, q)
        qc.compose(sub, qubits=list(range(k + 1)), inplace=True)
        amps = Statevector(qc).data.reshape(2, 2 ** (k + 1))   # rows: Bob, columns: Alice
        p = 0.0
        for i in range(R):                                     # successful outcomes
            bob = amps[:, i]
            pi = np.vdot(bob, bob).real
            p += pi
            if pi > 1e-12:
                worst_F = min(worst_F, abs(np.vdot(psi, bob)) ** 2 / pi)
        ps.append(p)
    return float(np.mean(ps)), float(np.ptp(ps)), worst_F

In [8]:
# ---------------------------------------------------------------------------
# Plot style shared by the figures (colours close to the paper)
# ---------------------------------------------------------------------------
plt.rcParams.update({"font.size": 10, "axes.grid": True, "grid.alpha": 0.3,
                     "legend.fontsize": 8, "figure.dpi": 110})
METHOD_STYLE = {"qsd": dict(color="#1f1f9e", marker="D", label="QSD"),
                "isometry": dict(color="#22b8e6", marker="o", label="Isometry")}
K_COLORS = {1: "#0b1a6e", 2: "#1f78d1", 3: "#3ec7e0", 4: "#86d9c6", 5: "#2a9d8f"}

print("core loaded")

core loaded
